In [221]:
import gridstatus
import pandas as pd
from datetime import timedelta

In [205]:
caiso = gridstatus.CAISO()

In [227]:
date = pd.to_datetime("2022-10-17 00:00:00")
locations = ["DLAP_SDGE-APND"]

rt_df = caiso.get_lmp(date = date, market = "REAL_TIME_5_MIN", locations = locations, sleep = 5)
hourly_df = caiso.get_lmp(date = date, market = "DAY_AHEAD_HOURLY", locations = locations, sleep = 5)

2026-01-26 21:21:43 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_INTVL_LMP', 'version': 3}, 'params': {'market_run_id': 'RTM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-01-26 21:21:43 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6&queryname=PRC_INTVL_LMP&version=3&market_run_id=RTM&node=DLAP_SDGE-APND&startdatetime=20221017T07:00-0000&enddatetime=20221018T07:00-0000
2026-01-26 21:21:44 - DEBUG - Found 1 files: ['20221017_20221018_PRC_INTVL_LMP_RTM_20260126_21_21_43_v3.csv']
2026-01-26 21:21:44 - DEBUG - Parsing file: 20221017_20221018_PRC_INTVL_LMP_RTM_20260126_21_21_43_v3.csv
2026-01-26 21:21:49 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_LMP', 'version': 12}, 'params': {'market_run_id': 'DAM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-01-26 21:21:49 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip

In [231]:
rt_df_fixed = rt_df[["Time", "LMP"]].copy()
hourly_df_fixed = hourly_df[["Time", "LMP"]].copy()

In [217]:
rt_df_fixed["Year"] = rt_df_fixed["Time"].dt.year
rt_df_fixed["Month"] = rt_df_fixed["Time"].dt.month
rt_df_fixed["Day"] = rt_df_fixed["Time"].dt.day
rt_df_fixed["Hour"] = rt_df_fixed["Time"].dt.hour

rt_df_fixed = rt_df_fixed[["Month", "Day", "Hour", "LMP", "Year"]]

rt_df_fixed = rt_df_fixed.groupby(["Month", "Day", "Year", "Hour"], as_index=False).agg({"LMP": "mean"})

rt_df_fixed.rename(columns={"LMP": "RT5M_LMP"}, inplace=True)

In [219]:
hourly_df_fixed["Year"] = hourly_df_fixed["Time"].dt.year
hourly_df_fixed["Month"] = hourly_df_fixed["Time"].dt.month
hourly_df_fixed["Day"] = hourly_df_fixed["Time"].dt.day
hourly_df_fixed["Hour"] = hourly_df_fixed["Time"].dt.hour

hourly_df_fixed = hourly_df_fixed[["Month", "Day", "Hour", "LMP", "Year"]]

hourly_df_fixed.rename(columns={"LMP": "DAM_LMP"}, inplace=True)

In [189]:
merged_lmps = rt_df_fixed.merge(hourly_df_fixed, on= ["Year", "Month", "Day", "Hour"])

In [193]:
merged_lmps["RT5M-DAM"] = merged_lmps["avg_RT5M_LMP"] - merged_lmps["DAM_LMP"]

In [195]:
merged_lmps

,Month,Day,Year,Hour,RT5M_LMP,DAM_LMP,RT5M-DAM
0,10,12,2022,0,65.223515,71.10430,-5.880785
1,10,12,2022,1,63.019754,68.11263,-5.092876
2,10,12,2022,2,63.118109,66.91266,-3.794551
3,10,12,2022,3,63.521435,66.96994,-3.448505
4,10,12,2022,4,66.549274,67.19150,-0.642226
5,10,12,2022,5,69.815758,78.19385,-8.378092
6,10,12,2022,6,71.932236,95.40257,-23.470334
7,10,12,2022,7,91.015072,91.25000,-0.234928
8,10,12,2022,8,81.945302,65.59000,16.355302
9,10,12,2022,9,91.801483,70.00000,21.801483


In [318]:
start_dates = [pd.to_datetime("2022-10-17 00:00:00"), pd.to_datetime("2022-11-17 00:00:00"), pd.to_datetime("2022-12-18 00:00:00")]
end_dates = [pd.to_datetime("2022-11-17 00:00:00") , pd.to_datetime("2022-12-18 00:00:00"), pd.to_datetime("2023-01-01 00:00:00")]

location = ["DLAP_SDGE-APND"]

merged_lmp_chunks = []

for start_date, end_date in zip(start_dates, end_dates):

    rt_df = caiso.get_lmp(date=start_date, end=end_date, market="REAL_TIME_5_MIN", locations=location, sleep=5)
    hourly_df = caiso.get_lmp(date=start_date, end=end_date, market="DAY_AHEAD_HOURLY", locations=location, sleep=5)

    rt_df = rt_df[["Time", "LMP"]].copy()
    hourly_df = hourly_df[["Time", "LMP"]].copy()

    rt_df["Year"] = pd.to_datetime(rt_df["Time"]).dt.year
    rt_df["Month"] = pd.to_datetime(rt_df["Time"]).dt.month
    rt_df["Day"] = pd.to_datetime(rt_df["Time"]).dt.day
    rt_df["Hour"] = pd.to_datetime(rt_df["Time"]).dt.hour

    rt_df = rt_df[["Month", "Day", "Hour", "LMP", "Year"]]
    rt_df = rt_df.groupby(["Month", "Day", "Year", "Hour"], as_index=False).agg({"LMP": "mean"})
    rt_df["Time"] = pd.to_datetime(rt_df[["Year", "Month", "Day", "Hour"]])

    rt_df.rename(columns={"LMP": "RT5M_AVG_LMP"}, inplace=True)

    hourly_df["Time"] = pd.to_datetime(hourly_df["Time"]).dt.tz_localize(None)
    hourly_df.rename(columns={"LMP": "DAM_LMP"}, inplace=True)

    merged_lmps = rt_df.merge(hourly_df, on=["Time"], how="outer")
    merged_lmp_chunks.append(merged_lmps)

final_merged_lmp = pd.concat(merged_lmp_chunks, ignore_index=True)

  0%|                                                     | 0/2 [00:00<?, ?it/s]2026-01-26 22:36:05 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_INTVL_LMP', 'version': 3}, 'params': {'market_run_id': 'RTM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-01-26 22:36:05 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6&queryname=PRC_INTVL_LMP&version=3&market_run_id=RTM&node=DLAP_SDGE-APND&startdatetime=20221017T07:00-0000&enddatetime=20221117T07:00-0000
2026-01-26 22:37:00 - DEBUG - Found 1 files: ['20221017_20221116_PRC_INTVL_LMP_RTM_20260126_22_36_05_v3.csv']
2026-01-26 22:37:00 - DEBUG - Parsing file: 20221017_20221116_PRC_INTVL_LMP_RTM_20260126_22_36_05_v3.csv
 50%|██████████████████████▌                      | 1/2 [01:00<01:00, 60.39s/it]2026-01-26 22:37:05 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_INTVL_LMP', 'version': 3}, 'params': {

In [324]:
final_merged_lmp.groupby(["Time"], as_index=False).agg({"RT5M_AVG_LMP": "mean",	"DAM_LMP": "mean"})

,Time,RT5M_AVG_LMP,DAM_LMP
0,2022-10-17 00:00:00,72.001627,71.35753
1,2022-10-17 01:00:00,66.541725,66.63932
2,2022-10-17 02:00:00,65.459888,66.45573
3,2022-10-17 03:00:00,61.904599,66.66230
4,2022-10-17 04:00:00,71.945210,66.94720
...,...,...,...
1819,2022-12-31 19:00:00,104.332207,138.49821
1820,2022-12-31 20:00:00,101.484582,134.27605
1821,2022-12-31 21:00:00,105.550937,127.00922
1822,2022-12-31 22:00:00,116.745401,120.27866


In [332]:
final_merged_lmp.to_csv("../data/processed/caiso_lmps.csv", index=False)